In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from netCDF4 import Dataset

from scintill_ai.io import get_magnetometer_data, get_solar_data, get_solar_wind_data, get_aggregated_gnss_data
from scintill_ai.preprocess import get_solar_position
from var import START_DATE, END_DATE, DATA_IN, SMAG_YEARS, SMAG_STATIONS

## INTERMAGNET

In [ ]:
# df_kou = get_magnetometer_data(
#     Path(DATA_IN, 'KOU')
# ).loc[START_DATE:END_DATE, 'h']

# df_ttb = get_magnetometer_data(
#     Path(DATA_IN, 'TTB')
# ).loc[START_DATE:END_DATE, 'h']

In [ ]:
# df_mag = pd.merge(
#     left=df_kou,
#     right=df_ttb,
#     how='inner',
#     left_index=True,
#     right_index=True,
#     suffixes=['_kou', '_ttb']
# )

## SuperMAG

In [ ]:
def get_magnetometer_data(year: str, stations_list: list) -> pd.DataFrame:
    filepath = Path(DATA_IN, 'supermag', f'all_stations_all{year}.netcdf')

    # Extract only specified stations
    with Dataset(filepath, mode='r') as f:
        data = f.variables['id'][:1]
        df_ids = pd.DataFrame(data)
    
    cols_to_use = df_ids.iloc[0][df_ids.iloc[0].isin(stations_list)].index.tolist()
    cols_name = df_ids.iloc[0][df_ids.iloc[0].isin(stations_list)].tolist()
    cols_name = [name.lower() for name in cols_name]

    # Extract timestamps
    dfs_datetime = {}
    
    with Dataset(filepath, mode='r') as f:
        for var_ in ['time_dy', 'time_hr', 'time_mo', 'time_mt', 'time_sc', 'time_yr']:
            data = f.variables[var_][:]
            dfs_datetime[var_] = pd.DataFrame(data, columns=[var_])
    
    df_datetime = pd.concat(dfs_datetime.values(), axis=1)
    df_datetime['time_yr'] = df_datetime['time_yr'].astype(str)
    df_datetime['time_dy'] = df_datetime['time_dy'].astype(str).str.zfill(2)
    df_datetime['time_mo'] = df_datetime['time_mo'].astype(str).str.zfill(2)
    df_datetime['time_hr'] = df_datetime['time_hr'].astype(str).str.zfill(2)
    df_datetime['time_mt'] = df_datetime['time_mt'].astype(str).str.zfill(2)
    
    df_datetime['datetime'] = (
        df_datetime['time_yr'] + '-' + df_datetime['time_mo'] + '-' + df_datetime['time_dy'] + ' ' + df_datetime['time_hr'] + ':' + df_datetime['time_mt']
    )
    
    df_datetime['datetime'] = pd.to_datetime(
        df_datetime['datetime'],
        errors='coerce',
        format='%Y-%m-%d %H:%M',
    )

    # Extract actual magnetometer data
    dfs = {}
    variables = ['dbe_geo', 'dbn_geo'] # + ['sza']
    
    with Dataset(filepath, mode='r') as f:
        for var_ in variables:
            data = f.variables[var_][:, cols_to_use]
            df_var = pd.DataFrame(data, columns=cols_name)
            df_var.columns = [f"{var_}_{col}" for col in df_var.columns]
            dfs[var_] = df_var
    
    df = pd.concat(dfs.values(), axis=1).set_index(df_datetime['datetime'])
    
    for stat_ in cols_name:
        df[f'h_{stat_}'] = np.sqrt(df[f'dbe_geo_{stat_}']**2 + df[f'dbn_geo_{stat_}']**2)

    return df[[f'h_{stat_}' for stat_ in cols_name]]

In [ ]:
dfs = {}

for yr_ in SMAG_YEARS:
    dfs[yr_] = get_magnetometer_data(year=str(yr_), stations_list=SMAG_STATIONS)

df = pd.concat(dfs.values(), axis=0)
df['h_tmk'] = df['h_ttb'] - df['h_kou']

In [ ]:
# df_plt = df#.loc['2024-05-09':'2024-05-12']

# fig, (ax1, ax2) = plt.subplots(figsize=(21, 14), nrows=2, ncols=1, sharex=True)

# for i, stat_ in enumerate(SMAG_STATIONS):
#     ax = ax1 if i == 0 else ax2
#     ax.plot(df_plt.index, df_plt[f'h_{stat_.lower()}'], color='tab:blue')
    
#     [ax.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]

#     perc_na = df_plt[f'h_{stat_.lower()}'].isna().sum() / df_plt.shape[0]
#     ax.set_ylabel('Horizontal component [nT]')
#     ax.set_ylim(0)
#     ax.set_title(f'{stat_} magnetometer (NaN values: {perc_na:.1%})', fontsize=16, fontweight='bold')
#     ax.grid(True, axis='y', linewidth=0.3, alpha=0.8)

# plt.tight_layout()
# # plt.savefig('magnetometers.png', dpi=500)
# plt.show()

In [ ]:
df_plt = df.loc['2024-01-01':'2024-01-10']

fig, ax = plt.subplots(figsize=(21, 7))

ax.plot(df_plt.index, df_plt['h_tmk'], color='tab:blue')
[ax.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]

perc_na = df_plt['h_tmk'].isna().sum() / df_plt.shape[0]
ax.set_ylabel('Horizontal component [nT]')
# ax.set_ylim(0)
ax.set_title(f'TTB - KOU (NaN values: {perc_na:.1%})', fontsize=16, fontweight='bold')
ax.grid(True, axis='y', linewidth=0.3, alpha=0.8)

plt.tight_layout()
plt.savefig('tmk_magnetometers.png', dpi=500)
plt.show()

## GFZ

In [ ]:
df_solar = get_solar_data(START_DATE, END_DATE)

## OMNIweb

In [ ]:
df_omni = get_solar_wind_data(Path(DATA_IN, 'omniweb')).loc[
    START_DATE:END_DATE,
    ['field_magnitude_avg', 'wind_speed', 'wind_density', 'wind_pressure', 'eletric_field']
]

## ISMR

$$ S_{4,\hspace{0.15 em}\mathrm{denoised}} = \mathrm{Re}\left( \sqrt{S_4^2 - S_{4,\hspace{0.15 em}\mathrm{noise}}^2} \right) $$

In [ ]:
start = "2024-02-01"
end = "2024-02-02"
station_name = "PRU2"
field_list = "time_utc, svid, azim, elev, s4, s4_correction, locktime_l1"

df_gnss = get_aggregated_gnss_data(
    start=start,
    end=end,
    station_name=station_name,
    fields=field_list,
)

In [ ]:
df_gnss

## Solar Zenith Angle

In [ ]:
# get_solar_position(
#     df_XXX.index, columns=['zenith'], altitude=0,
# ).round(1)